In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

#wczytaj plik 'heart.csv'
df = pd.read_csv('../heart.csv')

In [ ]:
data = np.asarray(df)

X = data[:,:-1]
y =  data[:,-1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state=42)
model = SVC(kernel = 'linear')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## Inspekcja liniowego modelu SVM

In [ ]:
# Wagi cech (tylko dla kernel='linear')
# model.coef_ ma kształt (1, n_features) dla problemu binarnego
weights = model.coef_[0]
feature_names = df.columns[:-1]

print("Waga każdej cechy:")
for name, w in zip(feature_names, weights):
    print(f"  {name:20s}: {w:.4f}")

## Inspekcja SVM z jądrem nieliniowym (RBF, poly)

Dla jąder innych niż liniowe `coef_` **nie istnieje**. Alternatywnie możemy zastosować metodę `permutation_importance` z biblioteki `sklearn.inspection`, która mierzy spadek dokładności modelu po przetasowaniu wartości danej cechy. Im większy spadek, tym ważniejsza jest dana cecha dla modelu.

In [ ]:
from sklearn.inspection import permutation_importance

model_rbf = SVC(kernel='poly')
model_rbf.fit(X_train, y_train)

result = permutation_importance(model_rbf, X_test, y_test, n_repeats=30, random_state=42)

plt.figure(figsize=(10, 5))
sorted_idx = result.importances_mean.argsort()
plt.barh(feature_names[sorted_idx], result.importances_mean[sorted_idx],
         xerr=result.importances_std[sorted_idx])
plt.xlabel("Średni spadek dokładności po permutacji")
plt.title("Permutation Importance – SVM Poly")
plt.tight_layout()
plt.show()